In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score, RandomizedSearchCV,
                                      GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (f1_score, roc_auc_score, classification_report,
                              confusion_matrix, precision_recall_curve)
from catboost import CatBoostClassifier
from scipy.stats import randint, uniform

print("✅ Libraries berhasil diimport")

In [ ]:
df = pd.read_excel("E_Commerce_Dataset.xlsx", sheet_name="E Comm")
print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}")
df.head()

In [ ]:
# Distribusi target
print("=== Distribusi Target (Churn) ===")
print(df['Churn'].value_counts())
print(f"\nRasio Churn: {df['Churn'].mean()*100:.1f}%")

# Missing values
print("\n=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
# Visualisasi distribusi target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie(df['Churn'].value_counts(), labels=['No Churn','Churn'],
            autopct='%1.1f%%', colors=['#38ef7d','#ff416c'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title('Distribusi Churn', fontweight='bold')

# Churn by Tenure
avg_tenure = df.groupby('Churn')['Tenure'].mean()
axes[1].bar(['No Churn','Churn'], avg_tenure.values,
            color=['#38ef7d','#ff416c'], edgecolor='white', linewidth=1.5)
axes[1].set_title('Rata-rata Tenure vs Churn', fontweight='bold')
axes[1].set_ylabel('Rata-rata Tenure (bulan)')
for i, v in enumerate(avg_tenure.values):
    axes[1].text(i, v+0.2, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap korelasi numerik
fig, ax = plt.subplots(figsize=(14, 10))
num_df = df.select_dtypes(include='number')
corr   = num_df.corr()
mask   = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size':8})
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Standardisasi nilai kategorikal yang inkonsisten
df['PreferredPaymentMode'] = df['PreferredPaymentMode'].replace({
    'COD': 'Cash on Delivery', 'CC': 'Credit Card'
})
df['PreferredLoginDevice'] = df['PreferredLoginDevice'].replace({
    'Phone': 'Mobile Phone'
})

print("PaymentMode unique:", df['PreferredPaymentMode'].unique())
print("LoginDevice unique:", df['PreferredLoginDevice'].unique())

In [ ]:
# Definisi fitur dan target
FEATURES = [
    'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome',
    'PreferredPaymentMode', 'Gender', 'HourSpendOnApp',
    'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore',
    'MaritalStatus', 'NumberOfAddress', 'Complain',
    'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount',
    'DaySinceLastOrder', 'CashbackAmount'
]
NUM_COLS = [
    'Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp',
    'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress',
    'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed',
    'OrderCount', 'DaySinceLastOrder', 'CashbackAmount'
]
CAT_COLS = [
    'PreferredLoginDevice', 'PreferredPaymentMode', 'Gender',
    'PreferedOrderCat', 'MaritalStatus'
]

X = df[FEATURES]
y = df['Churn']

# Pipeline preprocessor
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1)),
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, NUM_COLS),
    ('cat', cat_pipe, CAT_COLS),
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"\nDistribusi y_train: {dict(y_train.value_counts())}")
print(f"Distribusi y_test : {dict(y_test.value_counts())}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

baseline_models = {
    'Logistic Regression':    LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':          RandomForestClassifier(n_estimators=100, random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingClassifier(random_state=42),
    'XGBoost':                XGBClassifier(eval_metric='logloss', verbosity=0, random_state=42),
    'LightGBM':               LGBMClassifier(verbose=-1, random_state=42),
    'CatBoost':               CatBoostClassifier(verbose=0, random_state=42,
                                                  auto_class_weights='Balanced'),
}

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline_results = []

for name, mdl in baseline_models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', mdl)])
    cv_f1 = cross_val_score(pipe, X_train, y_train,
                            cv=cv_strat, scoring='f1', n_jobs=-1)
    pipe.fit(X_train, y_train)
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:,1]
    baseline_results.append({
        'Model': name,
        'CV F1 (mean)': round(cv_f1.mean(), 4),
        'CV F1 (std)':  round(cv_f1.std(),  4),
        'Test F1':      round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':      round(roc_auc_score(y_test, y_proba), 4),
    })
    print(f"✅ {name:<28} CV F1: {cv_f1.mean():.4f} | Test F1: {f1_score(y_test,y_pred):.4f}")

df_baseline = pd.DataFrame(baseline_results).sort_values('Test F1', ascending=False)
df_baseline

In [ ]:
# Visualisasi baseline
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#667eea' if m != 'CatBoost' else '#ff416c'
          for m in df_baseline['Model']]
bars = ax.barh(df_baseline['Model'], df_baseline['Test F1'],
               color=colors, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, df_baseline['Test F1']):
    ax.text(bar.get_width()+0.002, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold')
ax.set_title('Baseline: Test F1 Score per Model', fontsize=13, fontweight='bold')
ax.set_xlabel('F1 Score')
ax.axvline(df_baseline['Test F1'].max(), color='gold', lw=2, ls='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# STAGE 1 — Wide RandomizedSearch
param_wide = {
    'model__iterations':          randint(100, 1000),
    'model__depth':               randint(3, 10),
    'model__learning_rate':       uniform(0.01, 0.29),
    'model__l2_leaf_reg':         uniform(1, 9),
    'model__random_strength':     uniform(0.1, 2.9),
    'model__bagging_temperature': uniform(0, 2),
    'model__border_count':        [32, 64, 128, 254],
    'model__grow_policy':         ['SymmetricTree', 'Depthwise', 'Lossguide'],
    'model__subsample':           uniform(0.5, 0.5),
    'model__colsample_bylevel':   uniform(0.5, 0.5),
    'model__auto_class_weights':  ['Balanced', 'SqrtBalanced'],
}

pipeline_cat = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', CatBoostClassifier(verbose=0, random_state=42,
                                  eval_metric='F1', od_type='Iter', od_wait=50))
])

stage1 = RandomizedSearchCV(
    pipeline_cat, param_distributions=param_wide,
    n_iter=50, cv=cv_strat, scoring='f1',
    n_jobs=-1, random_state=42, refit=True, verbose=1, error_score=0
)
stage1.fit(X_train, y_train)

best_s1    = stage1.best_params_
best_f1_s1 = stage1.best_score_
print(f"\n✅ Stage 1 Best F1 (CV): {best_f1_s1:.4f}")
print(f"   Best Params: {best_s1}")

In [ ]:
# STAGE 2 — Narrow GridSearch di sekitar nilai terbaik S1
def narrow_range(val, step, n=1, min_val=None, max_val=None, as_int=False):
    vals = [val + i * step for i in range(-n, n + 1)]
    if min_val is not None: vals = [v for v in vals if v >= min_val]
    if max_val is not None: vals = [v for v in vals if v <= max_val]
    if as_int: return sorted(set(int(round(v)) for v in vals))
    return sorted(set(round(v, 4) for v in vals))

s1_iter  = int(best_s1.get('model__iterations', 300))
s1_depth = int(best_s1.get('model__depth', 6))
s1_lr    = float(best_s1.get('model__learning_rate', 0.05))
s1_l2    = float(best_s1.get('model__l2_leaf_reg', 3))
s1_rs    = float(best_s1.get('model__random_strength', 1))
s1_bt    = float(best_s1.get('model__bagging_temperature', 1))
s1_border= best_s1.get('model__border_count', 64)
s1_grow  = best_s1.get('model__grow_policy', 'SymmetricTree')
s1_sub   = float(best_s1.get('model__subsample', 0.8))
s1_col   = float(best_s1.get('model__colsample_bylevel', 0.8))
s1_acw   = best_s1.get('model__auto_class_weights', 'Balanced')
if s1_acw is None: s1_acw = 'Balanced'

param_narrow = {
    'model__iterations':          narrow_range(s1_iter, 100, n=1, min_val=50, as_int=True),
    'model__depth':               narrow_range(s1_depth, 1, n=1, min_val=2, max_val=12, as_int=True),
    'model__learning_rate':       narrow_range(s1_lr, 0.01, n=1, min_val=0.005, max_val=0.3),
    'model__l2_leaf_reg':         narrow_range(s1_l2, 1.0, n=1, min_val=0.5),
    'model__random_strength':     [round(s1_rs, 2)],
    'model__bagging_temperature': [round(s1_bt, 2)],
    'model__border_count':        [s1_border if s1_border in [32,64,128,254] else 64],
    'model__grow_policy':         [s1_grow],
    'model__subsample':           [round(s1_sub, 2)],
    'model__colsample_bylevel':   [round(s1_col, 2)],
    'model__auto_class_weights':  [s1_acw],
}

total = 1
for v in param_narrow.values(): total *= len(v)
print(f"Total kombinasi Stage 2: {total} ({total*5} fits)")

stage2 = GridSearchCV(
    pipeline_cat, param_grid=param_narrow,
    cv=cv_strat, scoring='f1',
    n_jobs=-1, refit=True, verbose=1, error_score=0
)
stage2.fit(X_train, y_train)

best_s2    = stage2.best_params_
best_f1_s2 = stage2.best_score_
print(f"\n✅ Stage 2 Best F1 (CV): {best_f1_s2:.4f}")
print(f"   Peningkatan dari S1  : +{best_f1_s2 - best_f1_s1:.4f}")

# Pilih stage terbaik
if best_f1_s2 >= best_f1_s1:
    final_model = stage2.best_estimator_
    best_params = best_s2
    best_f1_cv  = best_f1_s2
    best_stage  = 2
else:
    final_model = stage1.best_estimator_
    best_params = best_s1
    best_f1_cv  = best_f1_s1
    best_stage  = 1

print(f"\n🏆 Gunakan hasil Stage {best_stage} — CV F1: {best_f1_cv:.4f}")

In [ ]:
thresholds  = np.arange(0.20, 0.71, 0.01)
oof_proba   = np.zeros(len(X_train))
oof_true    = np.zeros(len(X_train))

for fold, (tr_idx, val_idx) in enumerate(cv_strat.split(X_train, y_train)):
    clone = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', CatBoostClassifier(
            **{k.replace('model__',''):v for k,v in best_params.items()},
            verbose=0, random_state=42
        ))
    ])
    clone.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    oof_proba[val_idx] = clone.predict_proba(X_train.iloc[val_idx])[:,1]
    oof_true[val_idx]  = y_train.iloc[val_idx]
    print(f"  Fold {fold+1}/5 ✓")

f1_per_thresh = [
    f1_score(oof_true, (oof_proba>=t).astype(int), zero_division=0)
    for t in thresholds
]
best_idx       = np.argmax(f1_per_thresh)
best_threshold = round(thresholds[best_idx], 2)

# Plot threshold vs F1
plt.figure(figsize=(10, 4))
plt.plot(thresholds, f1_per_thresh, color='#667eea', linewidth=2)
plt.axvline(best_threshold, color='red', ls='--', lw=1.5,
            label=f'Optimal: {best_threshold}')
plt.axvline(0.5, color='gray', ls=':', lw=1.5, label='Default: 0.5')
plt.xlabel('Threshold'); plt.ylabel('OOF F1 Score')
plt.title('Threshold vs F1 Score (OOF)', fontweight='bold')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"\n✅ Optimal Threshold: {best_threshold}")
print(f"   OOF F1 @ {best_threshold}: {max(f1_per_thresh):.4f}")
print(f"   OOF F1 @ 0.50    : {f1_score(oof_true,(oof_proba>=0.5).astype(int)):.4f}")

In [ ]:
# Retrain pada full data (X, y)
final_model.fit(X, y)

y_proba = final_model.predict_proba(X_test)[:,1]
y_pred  = (y_proba >= best_threshold).astype(int)
test_f1 = f1_score(y_test, y_pred)
test_auc= roc_auc_score(y_test, y_proba)

print(f"Test F1      : {test_f1:.4f}")
print(f"Test ROC-AUC : {test_auc:.4f}")
print(f"Threshold    : {best_threshold}")
print(f"\n{classification_report(y_test, y_pred, target_names=['No Churn','Churn'])}") 

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn','Churn'],
            yticklabels=['No Churn','Churn'],
            linewidths=1, annot_kws={'size':13,'weight':'bold'})
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# Feature importance
cat_model = final_model.named_steps['model']
feat_names = NUM_COLS + CAT_COLS
fi = pd.Series(cat_model.get_feature_importance(), index=feat_names)
fi = fi.sort_values(ascending=True).tail(12)
axes[1].barh(fi.index, fi.values,
             color=['#ff416c' if v == fi.max() else '#667eea' for v in fi.values])
axes[1].set_title('Feature Importance (CatBoost)', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout(); plt.show()
print(f"\n🏆 Top-3 Fitur: {fi.sort_values(ascending=False).head(3).index.tolist()}")

In [ ]:
bundle = {
    'model':       final_model,
    'threshold':   best_threshold,
    'features':    FEATURES,
    'num_cols':    NUM_COLS,
    'cat_cols':    CAT_COLS,
    'cv_f1_mean':  round(best_f1_cv, 4),
    'test_f1':     round(test_f1, 4),
    'test_auc':    round(test_auc, 4),
    'model_name':  'CatBoost',
    'best_params': best_params,
}

with open('model.pkl', 'wb') as f:
    pickle.dump(bundle, f)

import os
size_kb = os.path.getsize('model.pkl') / 1024
print(f"✅ model.pkl tersimpan ({size_kb:.1f} KB)")
print(f"   Model     : CatBoost (Stage {best_stage})")
print(f"   Threshold : {best_threshold}")
print(f"   Test F1   : {test_f1:.4f}")
print(f"   ROC-AUC   : {test_auc:.4f}")